# Аналитическая платформа интернет-магазина

Финальный проект: сквозной конвейер обработки данных — от трёх разрозненных CSV-файлов
до готовых аналитических выводов и экспорта отчётов.

**Схема данных и связи между источниками**

```
customers.csv ──┐
                ├── customer_id ──> orders.csv <── product_id ── products.csv
products.csv  ──┘
```

**Конвейер обработки**

```
подготовка файлов -> загрузка каталогов -> валидация (+ журнал отбраковки)
      -> объединение по ключу -> нормализация текста
      -> аналитика (чистый Python | NumPy | Pandas) -> отчёт -> JSON + CSV
```

**Содержание**

| Этап | Что делает |
|------|------------|
| 1 | Подготовка файлов данных (`save_dataset`) |
| 2 | Загрузка каталогов в словари (`load_products`, `load_customers`) |
| 3 | Валидация заказов и журнал отбраковки (`validate_orders`) |
| 4 | Объединение трёх источников по ключу (`enrich_orders`) |
| 5 | Нормализация текстовых полей (`normalize_orders`) |
| 6 | Аналитика на чистом Python |
| 7 | Статистика и поиск аномалий средствами NumPy |
| 8 | Многомерный анализ средствами Pandas |
| 9 | Сборка пайплайна и отчёты (`run_pipeline`, экспорт в JSON и CSV) |
| 10 | Интерактивная аналитическая консоль (`console`) |

В файле заказов намеренно заложены пять проблемных строк, которые платформа обязана
отбраковать, не прерывая работу: заказы 12, 15, 17, 19 и 21. Кроме того, в выручке
учитываются только заказы со статусом `completed`.

## Этап 1. Подготовка файлов данных

Функция `save_dataset(rows, filename)` записывает список строк в CSV-файл: через
конструкцию `with`, модулем `csv`, в кодировке `utf-8` и одним вызовом `writerows`.
С её помощью создаются все три исходных файла.

Эта же функция переиспользуется дальше — на этапе 3 ею записывается журнал
отбраковки, что избавляет от дублирования кода записи CSV.

In [1]:
import csv

# Имена файлов собраны в одном месте: пайплайн и консоль ссылаются на эти константы.
CUSTOMERS_FILE = "customers.csv"
PRODUCTS_FILE = "products.csv"
ORDERS_FILE = "orders.csv"
REJECTED_FILE = "rejected.csv"
SUMMARY_JSON = "summary.json"
CATEGORY_CSV = "revenue_by_category.csv"

CUSTOMERS_ROWS = [
    ["customer_id", "name", "city", "registration_date", "segment"],
    ["c1", "Анна Иванова", "Москва", "2025-11-02", "розница"],
    ["c2", "Петр Сидоров", "Санкт-Петербург", "2025-12-15", "розница"],
    ["c3", "Мария Петрова", "Москва", "2026-01-05", "опт"],
    ["c4", "Иван Кузнецов", "Казань", "2025-10-20", "розница"],
    ["c5", "Ольга Смирнова", "Москва", "2026-01-10", "опт"],
    ["c6", "Дмитрий Волков", "Новосибирск", "2025-09-30", "розница"],
    ["c7", "Елена Морозова", "Санкт-Петербург", "2026-01-12", "опт"],
    ["c8", "Сергей Попов", "Казань", "2025-12-01", "розница"],
]

PRODUCTS_ROWS = [
    ["product_id", "name", "category", "unit_price", "in_stock"],
    ["p1", "Python для начинающих", "Книги", 650, "yes"],
    ["p2", "Наушники", "Электроника", 2300, "yes"],
    ["p3", "Чистый код", "Книги", 900, "yes"],
    ["p4", "Мышь", "Электроника", 550, "yes"],
    ["p5", "Алгоритмы", "Книги", 1200, "no"],
    ["p6", "Футболка", "Одежда", 800, "yes"],
    ["p7", "Джинсы", "Одежда", 2500, "yes"],
    ["p8", "Клавиатура", "Электроника", 1500, "yes"],
    ["p9", "Куртка", "Одежда", 4500, "yes"],
    ["p10", "Грокаем алгоритмы", "Книги", 1100, "yes"],
]

ORDERS_ROWS = [
    ["order_id", "order_date", "customer_id", "product_id", "quantity", "status"],
    [1, "2026-01-10", "c1", "p1", 2, "completed"],
    [2, "2026-01-10", "c2", "p2", 1, "completed"],
    [3, "2026-01-11", "c3", "p3", 5, "completed"],
    [4, "2026-01-11", "c1", "p4", 3, "completed"],
    [5, "2026-01-12", "c4", "p6", 2, "completed"],
    [6, "2026-01-12", "c2", "p8", 1, "cancelled"],
    [7, "2026-01-13", "c5", "p9", 1, "completed"],
    [8, "2026-01-13", "c3", "p7", 4, "completed"],
    [9, "2026-01-14", "c6", "p2", 2, "completed"],
    [10, "2026-01-14", "c1", "p3", 1, "completed"],
    [11, "2026-01-15", "c7", "p10", 10, "completed"],
    [12, "2026-01-15", "c4", "p4", "x", "completed"],        # нечисловое количество
    [13, "2026-01-16", "c8", "p6", 3, "pending"],
    [14, "2026-01-16", "c5", "p8", 2, "completed"],
    [15, "2026-01-17", "c2", "p1", "", "completed"],         # пустое количество
    [16, "2026-01-17", "c3", "p9", 2, "completed"],
    [17, "2026-01-18", "c6", "p99", 1, "completed"],         # несуществующий товар
    [18, "2026-01-18", "c1", "p2", 1, "completed"],
    [19, "2026-01-19", "c99", "p3", 2, "completed"],         # несуществующий покупатель
    [20, "2026-01-19", "c7", "p7", 3, "completed"],
    [21, "2026-01-20", "c4", "p8", -1, "completed"],         # отрицательное количество
    [22, "2026-01-20", "c8", "p4", 4, "cancelled"],
    [23, "2026-01-21", "c5", "p10", 6, "completed"],
    [24, "2026-01-21", "c3", "p2", 2, "completed"],
    [25, "2026-01-22", "c1", "p6", 1, "completed"],
    [26, "2026-01-22", "c2", "p3", 3, "completed"],
    [27, "2026-01-23", "c6", "p9", 1, "completed"],
    [28, "2026-01-23", "c7", "p8", 2, "completed"],
]


def save_dataset(rows, filename):
    """Записывает список строк в CSV-файл одним вызовом writerows.

    rows     — список строк (каждая строка — список или кортеж значений);
    filename — имя создаваемого файла.
    Возвращает имя файла, чтобы вызов можно было использовать в цепочке.
    """
    with open(filename, "w", encoding="utf-8", newline="") as file:
        writer = csv.writer(file)
        writer.writerows(rows)
    return filename


def prepare_files():
    """Этап 1 пайплайна: создаёт три исходных файла и возвращает их имена."""
    save_dataset(CUSTOMERS_ROWS, CUSTOMERS_FILE)
    save_dataset(PRODUCTS_ROWS, PRODUCTS_FILE)
    save_dataset(ORDERS_ROWS, ORDERS_FILE)
    return CUSTOMERS_FILE, PRODUCTS_FILE, ORDERS_FILE


# Демонстрация этапа 1
created = prepare_files()
for name, rows in zip(created, (CUSTOMERS_ROWS, PRODUCTS_ROWS, ORDERS_ROWS)):
    print(f"{name}: записано строк данных — {len(rows) - 1} (+ заголовок)")

print("\nПервые строки orders.csv:")
with open(ORDERS_FILE, encoding="utf-8") as file:
    for number, line in enumerate(file):
        if number > 3:
            break
        print("  " + line.rstrip())

customers.csv: записано строк данных — 8 (+ заголовок)
products.csv: записано строк данных — 10 (+ заголовок)
orders.csv: записано строк данных — 28 (+ заголовок)

Первые строки orders.csv:
  order_id,order_date,customer_id,product_id,quantity,status
  1,2026-01-10,c1,p1,2,completed
  2,2026-01-10,c2,p2,1,completed
  3,2026-01-11,c3,p3,5,completed


## Этап 2. Загрузка каталогов в словари

Каталоги читаются через `csv.DictReader` и складываются в словари вида
`{идентификатор: запись}`. Такая структура выбрана осознанно: поиск товара или
покупателя по идентификатору становится операцией за константное время, тогда как
по списку пришлось бы каждый раз делать полный перебор. Именно этим словарём
пользуются этапы 3 и 4.

Цена приводится к `float`, отсутствие файла перехватывается через `FileNotFoundError`
и не роняет программу.

In [2]:
def _load_catalog(filename, key_field, numeric_fields=()):
    """Общий загрузчик справочника: строит словарь {идентификатор: запись}.

    key_field      — поле-идентификатор, оно же ключ итогового словаря;
    numeric_fields — поля, которые нужно привести к float.
    Обобщённая функция избавляет load_products и load_customers от дублирования.
    """
    catalog = {}
    try:
        with open(filename, encoding="utf-8", newline="") as file:
            for row in csv.DictReader(file):
                for field in numeric_fields:
                    row[field] = float(row[field])
                catalog[row[key_field]] = row
    except FileNotFoundError:
        print(f"Файл {filename} не найден: справочник загружен пустым")
    return catalog


def load_products(filename):
    """Читает каталог товаров: {product_id: {...}}, цена приведена к float."""
    return _load_catalog(filename, "product_id", numeric_fields=("unit_price",))


def load_customers(filename):
    """Читает базу покупателей: {customer_id: {...}}."""
    return _load_catalog(filename, "customer_id")


# Демонстрация этапа 2
products = load_products(PRODUCTS_FILE)
customers = load_customers(CUSTOMERS_FILE)

print(f"Загружено товаров: {len(products)}, покупателей: {len(customers)}")
print("products['p2'] =", products["p2"])
print("customers['c3'] =", customers["c3"])

# Проверка обработки отсутствующего файла — программа продолжает работу
print("Нет файла:", load_products("missing_products.csv"))

Загружено товаров: 10, покупателей: 8
products['p2'] = {'product_id': 'p2', 'name': 'Наушники', 'category': 'Электроника', 'unit_price': 2300.0, 'in_stock': 'yes'}
customers['c3'] = {'customer_id': 'c3', 'name': 'Мария Петрова', 'city': 'Москва', 'registration_date': '2026-01-05', 'segment': 'опт'}
Файл missing_products.csv не найден: справочник загружен пустым
Нет файла: {}


## Этап 3. Валидация заказов и журнал отбраковки

`validate_orders` разделяет строки заказов на корректные и отбракованные. Проверки
идут по очереди, и первая же сработавшая причина отправляет строку в брак:

1. количество не преобразуется в число — перехватывается в `try / except`;
2. количество меньше либо равно нулю;
3. товара нет в каталоге (проверка через `in`);
4. покупателя нет в базе (проверка через `in`).

Причина хранится рядом с идентификатором заказа в **кортеже** — неизменяемой паре
«идентификатор и причина». Журнал отбраковки параллельно пишется в `rejected.csv`
уже имеющейся функцией `save_dataset`.

In [3]:
# Причины отбраковки вынесены в константы: один и тот же текст идёт и в журнал, и в отчёт
REASON_NOT_A_NUMBER = "нечисловое количество"
REASON_NOT_POSITIVE = "неположительное количество"
REASON_UNKNOWN_PRODUCT = "неизвестный товар"
REASON_UNKNOWN_CUSTOMER = "неизвестный покупатель"

REJECTED_HEADER = ("order_id", "reason")


def validate_orders(filename, products, customers, log_file=REJECTED_FILE):
    """Делит заказы на корректные и отбракованные.

    Возвращает кортеж (valid, rejected):
      valid    — список словарей-заказов с количеством, приведённым к int;
      rejected — список кортежей (order_id, причина).
    Отбракованные записи попутно сохраняются в CSV-журнал.
    """
    valid = []
    rejected = []

    try:
        with open(filename, encoding="utf-8", newline="") as file:
            rows = list(csv.DictReader(file))
    except FileNotFoundError:
        print(f"Файл заказов {filename} не найден: обрабатывать нечего")
        return valid, rejected

    for row in rows:
        order_id = row["order_id"]

        try:
            quantity = int(row["quantity"])
        except (ValueError, TypeError):
            # сюда попадают и "x", и пустая строка, и отсутствующее значение
            rejected.append((order_id, REASON_NOT_A_NUMBER))
            continue

        if quantity <= 0:
            rejected.append((order_id, REASON_NOT_POSITIVE))
            continue

        if row["product_id"] not in products:
            rejected.append((order_id, REASON_UNKNOWN_PRODUCT))
            continue

        if row["customer_id"] not in customers:
            rejected.append((order_id, REASON_UNKNOWN_CUSTOMER))
            continue

        order = dict(row)              # копия, чтобы не держать ссылку на строку чтения
        order["quantity"] = quantity   # количество уже числом
        valid.append(order)

    save_dataset([REJECTED_HEADER] + rejected, log_file)
    return valid, rejected


# Демонстрация этапа 3
total_rows = len(ORDERS_ROWS) - 1
valid, rejected = validate_orders(ORDERS_FILE, products, customers)

print(f"Всего строк заказов: {total_rows}")
print(f"Корректных заказов:  {len(valid)}")
print(f"Отбраковано:         {len(rejected)}")
print("\nЖурнал отбраковки:")
for order_id, reason in rejected:
    print(f"  заказ {order_id} — {reason}")

print(f"\nСодержимое {REJECTED_FILE}:")
with open(REJECTED_FILE, encoding="utf-8") as file:
    print(file.read().rstrip())

Всего строк заказов: 28
Корректных заказов:  23
Отбраковано:         5

Журнал отбраковки:
  заказ 12 — нечисловое количество
  заказ 15 — нечисловое количество
  заказ 17 — неизвестный товар
  заказ 19 — неизвестный покупатель
  заказ 21 — неположительное количество

Содержимое rejected.csv:
order_id,reason
12,нечисловое количество
15,нечисловое количество
17,неизвестный товар
19,неизвестный покупатель
21,неположительное количество


## Этап 4. Объединение источников данных

`enrich_orders` выполняет соединение трёх таблиц по ключу вручную — то, что в Pandas
делается методом `merge`. Для каждого корректного заказа из каталогов подтягиваются
название, категория и цена товара, а также имя, город и сегмент покупателя; сразу же
считается выручка заказа.

Поскольку этап 3 уже гарантировал существование идентификаторов, обращение к словарям
безопасно. Исходные каталоги при этом не изменяются: заказ копируется через `dict()`,
а из справочников берутся только значения.

In [4]:
def enrich_orders(valid_orders, products, customers):
    """Дополняет заказы данными из каталога товаров и базы покупателей.

    К каждому заказу добавляются product, category, price, customer, city, segment
    и вычисленная выручка revenue = quantity * price.
    Исходные заказы и каталоги не изменяются — возвращается новый список.
    """
    enriched = []

    for order in valid_orders:
        product = products[order["product_id"]]
        customer = customers[order["customer_id"]]

        row = dict(order)
        row["product"] = product["name"]
        row["category"] = product["category"]
        row["price"] = product["unit_price"]
        row["customer"] = customer["name"]
        row["city"] = customer["city"]
        row["segment"] = customer["segment"]
        row["revenue"] = row["quantity"] * row["price"]

        enriched.append(row)

    return enriched


# Демонстрация этапа 4
enriched = enrich_orders(valid, products, customers)

print(f"Обогащено заказов: {len(enriched)}")
print("\nenriched[1] =")
for key, value in enriched[1].items():
    print(f"  {key}: {value!r}")

print("\nКаталоги не изменились:", "revenue" not in products["p2"] and "revenue" not in customers["c2"])

Обогащено заказов: 23

enriched[1] =
  order_id: '2'
  order_date: '2026-01-10'
  customer_id: 'c2'
  product_id: 'p2'
  quantity: 1
  status: 'completed'
  product: 'Наушники'
  category: 'Электроника'
  price: 2300.0
  customer: 'Петр Сидоров'
  city: 'Санкт-Петербург'
  segment: 'розница'
  revenue: 2300.0

Каталоги не изменились: True


## Этап 5. Нормализация текстовых полей

Нормализация — обязательный защитный шаг: в реальных выгрузках одна и та же категория
легко встречается как `" книги"`, `"Книги"` и `"КНИГИ"`, и без приведения к единому виду
аналитика посчитает их разными группами.

Правила вынесены в словарь `NORMALIZATION_RULES`, поэтому добавить новое поле — значит
дописать одну строку, а не ещё одну ветку кода. Числовые поля (`quantity`, `price`,
`revenue`) не затрагиваются. Название товара очищается от пробелов, но не приводится
к `.title()`: это испортило бы написание вроде «Python для начинающих».

In [5]:
# Поле -> правило приведения. Числовые поля в таблицу намеренно не входят.
NORMALIZATION_RULES = {
    "customer": "title",   # имя покупателя: каждое слово с заглавной буквы
    "city": "title",       # город
    "category": "title",   # категория товара
    "product": "strip",    # название товара: только чистка пробелов
    "status": "lower",     # статус: единый регистр для надёжного сравнения
    "segment": "lower",    # сегмент: "Опт" и "опт" — одна и та же группа
}


def clean_text(value, rule):
    """Приводит строку к единому виду: убирает лишние пробелы и применяет правило.

    split() + join() схлопывают любые повторяющиеся пробелы внутри строки,
    а не только по краям, как это делает .strip().
    """
    if not isinstance(value, str):
        return value

    text = " ".join(value.split())
    if rule == "title":
        return text.title()
    if rule == "lower":
        return text.lower()
    return text


def normalize_orders(orders):
    """Приводит текстовые поля заказов к единому виду, не трогая числовые."""
    normalized = []

    for order in orders:
        row = dict(order)
        for field, rule in NORMALIZATION_RULES.items():
            if field in row:
                row[field] = clean_text(row[field], rule)
        normalized.append(row)

    return normalized


# Демонстрация этапа 5 на намеренно «грязной» записи
dirty = dict(enriched[1])
dirty.update({"customer": "  петр   сидоров ", "city": "санкт-петербург ",
              "category": " ЭЛЕКТРОНИКА", "status": " Completed "})

cleaned = normalize_orders([dirty])[0]
shown = ("customer", "city", "category", "status")
print("до нормализации:", {field: dirty[field] for field in shown})
print("после:          ", {field: cleaned[field] for field in shown})

enriched = normalize_orders(enriched)
print("\nЧисловые поля не затронуты:",
      enriched[1]["quantity"], enriched[1]["price"], enriched[1]["revenue"])

до нормализации: {'customer': '  петр   сидоров ', 'city': 'санкт-петербург ', 'category': ' ЭЛЕКТРОНИКА', 'status': ' Completed '}
после:           {'customer': 'Петр Сидоров', 'city': 'Санкт-Петербург', 'category': 'Электроника', 'status': 'completed'}

Числовые поля не затронуты: 1 2300.0 2300.0


## Этап 6. Аналитика на чистом Python

Все расчёты ведутся только по завершённым заказам: `cancelled` и `pending` в выручку
не попадают.

Ключевая функция здесь — `revenue_by(orders, field)`: имя поля передаётся параметром,
поэтому одна функция считает выручку и по категориям, и по городам, и по сегментам.
По той же причине `top_customers` и `top_products` построены на общем помощнике
`top_by`: они различаются только тем, какое поле группируется и какое суммируется.

In [6]:
COMPLETED_STATUS = "completed"


def completed_only(orders):
    """Оставляет только завершённые заказы — именно они формируют выручку."""
    return [order for order in orders if order["status"] == COMPLETED_STATUS]


def total_revenue(orders):
    """Суммарная выручка по переданным заказам."""
    return sum(order["revenue"] for order in orders)


def revenue_by(orders, field):
    """Выручка в разрезе произвольного поля: категории, города, сегмента и т. д.

    Возвращает словарь {значение поля: выручка}, отсортированный по убыванию суммы.
    """
    totals = {}
    for order in orders:
        key = order[field]
        totals[key] = totals.get(key, 0.0) + order["revenue"]
    return dict(sorted(totals.items(), key=lambda pair: pair[1], reverse=True))


def top_by(orders, key_field, value_field, n):
    """Топ-n значений key_field по сумме value_field. Основа для топов ниже."""
    totals = {}
    for order in orders:
        key = order[key_field]
        totals[key] = totals.get(key, 0) + order[value_field]
    ranking = sorted(totals.items(), key=lambda pair: pair[1], reverse=True)
    return ranking[:n]


def top_customers(orders, n=3):
    """Покупатели с наибольшей суммой покупок."""
    return top_by(orders, "customer", "revenue", n)


def top_products(orders, n=3):
    """Товары с наибольшим проданным количеством."""
    return top_by(orders, "product", "quantity", n)


def unique_values(orders, field):
    """Множество уникальных значений поля — множество само отбрасывает повторы."""
    return {order[field] for order in orders}


# Демонстрация этапа 6
completed = completed_only(enriched)

print("Завершённых заказов:", len(completed))
print("Общая выручка:", total_revenue(completed))
print()
print("revenue_by(completed, 'category') =", revenue_by(completed, "category"))
print("revenue_by(completed, 'city')     =", revenue_by(completed, "city"))
print("revenue_by(completed, 'segment')  =", revenue_by(completed, "segment"))
print()
print("top_customers(completed, 3) =", top_customers(completed, 3))
print("top_products(completed, 3)  =", top_products(completed, 3))
print()
print("Уникальных категорий:", sorted(unique_values(completed, "category")))
print("Уникальных городов:  ", sorted(unique_values(completed, "city")))
print("Уникальных покупателей:", len(unique_values(completed, "customer")))

Завершённых заказов: 20
Общая выручка: 86350.0

revenue_by(completed, 'category') = {'Одежда': 37900.0, 'Книги': 27000.0, 'Электроника': 21450.0}
revenue_by(completed, 'city')     = {'Москва': 49150.0, 'Санкт-Петербург': 26500.0, 'Новосибирск': 9100.0, 'Казань': 1600.0}
revenue_by(completed, 'segment')  = {'опт': 63700.0, 'розница': 22650.0}

top_customers(completed, 3) = [('Мария Петрова', 28100.0), ('Елена Морозова', 21500.0), ('Ольга Смирнова', 14100.0)]
top_products(completed, 3)  = [('Грокаем алгоритмы', 16), ('Чистый код', 9), ('Джинсы', 7)]

Уникальных категорий: ['Книги', 'Одежда', 'Электроника']
Уникальных городов:   ['Казань', 'Москва', 'Новосибирск', 'Санкт-Петербург']
Уникальных покупателей: 7


## Этап 7. Статистика и поиск аномалий средствами NumPy

Выручка завершённых заказов собирается в массив NumPy, по которому считаются средний
чек, медиана, стандартное отклонение и квартили.

Аномально крупным считается заказ, выручка которого превышает порог
`среднее + 2 × стандартное отклонение`. Отбор выполняется логической индексацией:
условие, применённое к массиву, даёт булеву маску, и этой маской сразу выбираются
нужные заказы — без явного цикла.

In [7]:
import numpy as np

ANOMALY_SIGMA = 2  # во сколько стандартных отклонений укладывается «нормальный» заказ


def revenue_array(orders):
    """Собирает выручку заказов в массив NumPy."""
    return np.array([order["revenue"] for order in orders], dtype=float)


def revenue_analytics(orders):
    """Статистика по выручке: средний чек, медиана, отклонение и квартили."""
    revenues = revenue_array(orders)
    if revenues.size == 0:
        return {name: 0.0 for name in ("mean", "median", "std", "q25", "q50", "q75")}

    return {
        "mean": round(float(revenues.mean()), 2),
        "median": round(float(np.median(revenues)), 2),
        "std": round(float(revenues.std()), 2),
        "q25": round(float(np.percentile(revenues, 25)), 2),
        "q50": round(float(np.percentile(revenues, 50)), 2),
        "q75": round(float(np.percentile(revenues, 75)), 2),
    }


def anomaly_threshold(orders, sigma=ANOMALY_SIGMA):
    """Порог аномалии: среднее плюс sigma стандартных отклонений."""
    revenues = revenue_array(orders)
    if revenues.size == 0:
        return 0.0
    return float(revenues.mean() + sigma * revenues.std())


def find_anomalies(orders, sigma=ANOMALY_SIGMA):
    """Аномально крупные заказы — выручка выше порога.

    Маска строится сравнением массива с порогом, и по ней выбираются заказы:
    отбор идёт логической индексацией, цикл не нужен.
    """
    revenues = revenue_array(orders)
    if revenues.size == 0:
        return []

    mask = revenues > revenues.mean() + sigma * revenues.std()
    orders_array = np.array(orders, dtype=object)   # одномерный массив словарей
    return list(orders_array[mask])


# Демонстрация этапа 7
stats = revenue_analytics(completed)
print("revenue_analytics(completed) =")
for name, value in stats.items():
    print(f"  {name}: {value}")

threshold = anomaly_threshold(completed)
anomalies = find_anomalies(completed)

print(f"\nПорог аномалии: {threshold:.2f}")
print(f"Аномальных заказов: {len(anomalies)}")
for order in anomalies:
    print(f"  заказ {order['order_id']}: {order['customer']}, "
          f"{order['product']} x{order['quantity']} — выручка {order['revenue']:.2f}")

revenue_analytics(completed) =
  mean: 4317.5
  median: 3750.0
  std: 2969.48
  q25: 2137.5
  q50: 3750.0
  q75: 5100.0

Порог аномалии: 10256.46
Аномальных заказов: 1
  заказ 11: Елена Морозова, Грокаем алгоритмы x10 — выручка 11000.00


## Этап 8. Многомерный анализ средствами Pandas

Тот же набор данных переносится в `DataFrame`, и сводная таблица «категория × сегмент»
строится одним вызовом `pivot_table`. Пропуски заполняются нулями.

Сводные суммы по категориям сравниваются с результатом ручного подсчёта из этапа 6:
совпадение значений подтверждает корректность обоих подходов.

In [8]:
import pandas as pd


def build_dataframe(orders):
    """Превращает список обогащённых заказов в DataFrame."""
    return pd.DataFrame(orders)


def category_segment_pivot(df):
    """Сводная таблица выручки: строки — категории, столбцы — сегменты."""
    return df.pivot_table(
        index="category",
        columns="segment",
        values="revenue",
        aggfunc="sum",
        fill_value=0,
    )


# Демонстрация этапа 8
df = build_dataframe(completed)
print("Размер таблицы:", df.shape)
print("Средняя выручка заказа:", round(df["revenue"].mean(), 2))
df.head()

Размер таблицы: (20, 13)
Средняя выручка заказа: 4317.5


,order_id,order_date,customer_id,product_id,quantity,status,product,category,price,customer,city,segment,revenue
0,1,2026-01-10,c1,p1,2,completed,Python для начинающих,Книги,650.0,Анна Иванова,Москва,розница,1300.0
1,2,2026-01-10,c2,p2,1,completed,Наушники,Электроника,2300.0,Петр Сидоров,Санкт-Петербург,розница,2300.0
2,3,2026-01-11,c3,p3,5,completed,Чистый код,Книги,900.0,Мария Петрова,Москва,опт,4500.0
3,4,2026-01-11,c1,p4,3,completed,Мышь,Электроника,550.0,Анна Иванова,Москва,розница,1650.0
4,5,2026-01-12,c4,p6,2,completed,Футболка,Одежда,800.0,Иван Кузнецов,Казань,розница,1600.0


In [9]:
pivot = category_segment_pivot(df)
print("Сводная таблица «категория × сегмент»:")
print(pivot)

print("\nРаспределение по категориям (groupby):")
by_category = df.groupby("category")["revenue"].agg(["sum", "mean", "count"]).round(2)
print(by_category)

Сводная таблица «категория × сегмент»:
segment          опт  розница
category                     
Книги        22100.0   4900.0
Одежда       31000.0   6900.0
Электроника  10600.0  10850.0

Распределение по категориям (groupby):
                 sum     mean  count
category                            
Книги        27000.0  4500.00      6
Одежда       37900.0  5414.29      7
Электроника  21450.0  3064.29      7


In [10]:
# Сверка двух подходов: ручной подсчёт из этапа 6 против groupby из Pandas
manual = revenue_by(completed, "category")
pandas_result = df.groupby("category")["revenue"].sum().to_dict()

print(f"{'Категория':<14}{'чистый Python':>16}{'Pandas':>12}   совпадает")
for category in sorted(manual):
    print(f"{category:<14}{manual[category]:>16.2f}{pandas_result[category]:>12.2f}"
          f"   {manual[category] == pandas_result[category]}")

assert manual == pandas_result, "Результаты двух подходов разошлись"
assert round(df["revenue"].mean(), 2) == revenue_analytics(completed)["mean"]
assert float(df["revenue"].sum()) == total_revenue(completed)
print("\nВсе сверки пройдены: ручная аналитика, NumPy и Pandas дают одинаковый результат")

Категория        чистый Python      Pandas   совпадает
Книги                 27000.00    27000.00   True
Одежда                37900.00    37900.00   True
Электроника           21450.00    21450.00   True

Все сверки пройдены: ручная аналитика, NumPy и Pandas дают одинаковый результат


## Этап 9. Сборка пайплайна и отчёты

`run_pipeline` читается как цепочка шагов: подготовка файлов → загрузка каталогов →
валидация → обогащение → нормализация → аналитика → статистика → отчёт. Результат
каждого шага передаётся следующему, а сводные показатели собираются в один словарь,
который и выводится в отчёт, и экспортируется в JSON.

Экспорт: `export_json` сохраняет сводку с `ensure_ascii=False`, чтобы кириллица осталась
читаемой, `export_csv` записывает агрегированную выручку по категориям модулем `csv`.
Текст отчёта дополнительно сохраняется в `report.txt`.

In [11]:
import json

REPORT_FILE = "report.txt"
CATEGORY_CSV_HEADER = ("key", "revenue")
TOP_N = 3


def build_summary(total_rows, rejected, valid, completed_orders):
    """Собирает все сводные показатели в один словарь."""
    stats = revenue_analytics(completed_orders)
    return {
        "total_rows": total_rows,
        "rejected": len(rejected),
        "valid": len(valid),
        "completed": len(completed_orders),
        "total_revenue": total_revenue(completed_orders),
        "average_check": stats["mean"],
        "median_check": stats["median"],
        "revenue_by_category": revenue_by(completed_orders, "category"),
        "revenue_by_segment": revenue_by(completed_orders, "segment"),
        "top_customers": top_customers(completed_orders, TOP_N),
        "anomalies": len(find_anomalies(completed_orders)),
    }


def format_report(summary):
    """Формирует итоговый текстовый отчёт из словаря сводных показателей."""
    lines = [
        "ОТЧЕТ АНАЛИТИЧЕСКОЙ ПЛАТФОРМЫ",
        f"Обработано строк заказов: {summary['total_rows']}",
        f"Отбраковано некорректных: {summary['rejected']}",
        f"Корректных заказов: {summary['valid']}",
        f"Завершенных заказов: {summary['completed']}",
        "",
        f"Общая выручка: {summary['total_revenue']:.2f}",
        f"Средний чек: {summary['average_check']:.2f}",
        f"Медианный чек: {summary['median_check']:.2f}",
        "",
        "Выручка по категориям:",
    ]
    lines += [f"  {key} — {value:.2f}" for key, value in summary["revenue_by_category"].items()]
    lines += ["", "Выручка по сегментам:"]
    lines += [f"  {key} — {value:.2f}" for key, value in summary["revenue_by_segment"].items()]
    lines += ["", "Топ покупателей:"]
    lines += [f"  {name} — {value:.2f}" for name, value in summary["top_customers"]]
    lines += ["", f"Аномальных заказов: {summary['anomalies']}"]
    return "\n".join(lines)


def export_json(summary, filename=SUMMARY_JSON):
    """Сохраняет сводные показатели в JSON с сохранением кириллицы."""
    with open(filename, "w", encoding="utf-8") as file:
        json.dump(summary, file, ensure_ascii=False, indent=4)
    return filename


def export_csv(rows, filename=CATEGORY_CSV, header=CATEGORY_CSV_HEADER):
    """Сохраняет агрегированные значения в CSV: заголовок плюс пары «ключ — сумма»."""
    with open(filename, "w", encoding="utf-8", newline="") as file:
        writer = csv.writer(file)
        writer.writerow(header)
        writer.writerows(rows)
    return filename


def export_report(report, filename=REPORT_FILE):
    """Сохраняет текст отчёта в файл."""
    with open(filename, "w", encoding="utf-8") as file:
        file.write(report)
    return filename


def run_pipeline(verbose=True):
    """Полный конвейер обработки: от подготовки файлов до сохранённых отчётов.

    Возвращает словарь-контекст со всеми промежуточными результатами,
    чтобы интерактивная консоль могла пользоваться ими без пересчёта.
    """
    customers_file, products_file, orders_file = prepare_files()

    products = load_products(products_file)
    customers = load_customers(customers_file)

    valid, rejected = validate_orders(orders_file, products, customers)
    enriched = normalize_orders(enrich_orders(valid, products, customers))
    completed = completed_only(enriched)

    total_rows = len(ORDERS_ROWS) - 1
    summary = build_summary(total_rows, rejected, valid, completed)
    report = format_report(summary)

    export_json(summary)
    export_csv(list(summary["revenue_by_category"].items()))
    export_report(report)

    if verbose:
        print(report)

    return {
        "products": products,
        "customers": customers,
        "valid": valid,
        "rejected": rejected,
        "enriched": enriched,
        "completed": completed,
        "summary": summary,
        "report": report,
        "analytics": revenue_analytics(completed),
        "anomalies": find_anomalies(completed),
        "dataframe": build_dataframe(completed),
    }


# Демонстрация этапа 9
context = run_pipeline()

ОТЧЕТ АНАЛИТИЧЕСКОЙ ПЛАТФОРМЫ
Обработано строк заказов: 28
Отбраковано некорректных: 5
Корректных заказов: 23
Завершенных заказов: 20

Общая выручка: 86350.00
Средний чек: 4317.50
Медианный чек: 3750.00

Выручка по категориям:
  Одежда — 37900.00
  Книги — 27000.00
  Электроника — 21450.00

Выручка по сегментам:
  опт — 63700.00
  розница — 22650.00

Топ покупателей:
  Мария Петрова — 28100.00
  Елена Морозова — 21500.00
  Ольга Смирнова — 14100.00

Аномальных заказов: 1


In [12]:
# Содержимое сохранённых файлов
print(f"--- {SUMMARY_JSON} ---")
with open(SUMMARY_JSON, encoding="utf-8") as file:
    print(file.read())

print(f"--- {CATEGORY_CSV} ---")
with open(CATEGORY_CSV, encoding="utf-8") as file:
    print(file.read().rstrip())

--- summary.json ---
{
    "total_rows": 28,
    "rejected": 5,
    "valid": 23,
    "completed": 20,
    "total_revenue": 86350.0,
    "average_check": 4317.5,
    "median_check": 3750.0,
    "revenue_by_category": {
        "Одежда": 37900.0,
        "Книги": 27000.0,
        "Электроника": 21450.0
    },
    "revenue_by_segment": {
        "опт": 63700.0,
        "розница": 22650.0
    },
    "top_customers": [
        [
            "Мария Петрова",
            28100.0
        ],
        [
            "Елена Морозова",
            21500.0
        ],
        [
            "Ольга Смирнова",
            14100.0
        ]
    ],
    "anomalies": 1
}
--- revenue_by_category.csv ---
key,revenue
Одежда,37900.0
Книги,27000.0
Электроника,21450.0


## Этап 10. Интерактивная аналитическая консоль

Данные готовятся пайплайном один раз до входа в цикл, поэтому ни один пункт меню
ничего не пересчитывает заново. Цикл `while` повторяет меню, пока не выбран выход;
на неверный ввод выводится сообщение, и работа продолжается.

Пункт «выручка по разрезу» опирается на обобщённую функцию `revenue_by` из этапа 6:
пользователь выбирает поле, а считает его та же самая функция.

Параметр `ask` по умолчанию равен встроенной `input`, но его можно подменить — это
позволяет показать работу консоли в блокноте, который должен выполняться целиком
без остановки на ручном вводе. Для живой работы достаточно вызвать `console()`.

In [13]:
MENU = """
=== АНАЛИТИЧЕСКАЯ КОНСОЛЬ ===
1 — полный отчёт
2 — выручка по разрезу
3 — топ покупателей и товаров
4 — аномальные заказы
5 — сохранить отчёты в файлы
0 — выход
""".strip()

# Разрез -> (поле заказа, подпись). Множество ключей используется для проверки ввода.
BREAKDOWNS = {
    "1": ("category", "категориям"),
    "2": ("city", "городам"),
    "3": ("segment", "сегментам"),
}
BREAKDOWN_CODES = set(BREAKDOWNS)


def print_revenue_breakdown(orders, ask):
    """Спрашивает разрез и печатает выручку по нему."""
    print("Разрезы: 1 — категории, 2 — города, 3 — сегменты")
    choice = ask("Выберите разрез: ").strip()

    if choice not in BREAKDOWN_CODES:
        print("Неизвестный разрез, возвращаюсь в меню")
        return

    field, title = BREAKDOWNS[choice]
    print(f"Выручка по {title}:")
    for key, value in revenue_by(orders, field).items():
        print(f"  {key} — {value:.2f}")


def console(ask=input, context=None):
    """Интерактивная аналитическая консоль.

    ask     — функция запроса ввода (по умолчанию input);
    context — готовый результат пайплайна; если не передан, пайплайн запускается сам.
    """
    if context is None:
        context = run_pipeline(verbose=False)

    completed = context["completed"]

    while True:
        print(MENU)
        choice = ask("Выберите пункт меню: ").strip()

        if choice == "0":
            print("Работа завершена")
            break

        elif choice == "1":
            print(context["report"])

        elif choice == "2":
            print_revenue_breakdown(completed, ask)

        elif choice == "3":
            print("Топ покупателей:")
            for name, value in top_customers(completed, TOP_N):
                print(f"  {name} — {value:.2f}")
            print("Топ товаров по количеству:")
            for name, quantity in top_products(completed, TOP_N):
                print(f"  {name} — {quantity} шт.")

        elif choice == "4":
            print(f"Порог аномалии: {anomaly_threshold(completed):.2f}")
            for order in context["anomalies"]:
                print(f"  заказ {order['order_id']}: {order['customer']}, "
                      f"{order['product']} — {order['revenue']:.2f}")

        elif choice == "5":
            export_json(context["summary"])
            export_csv(list(context["summary"]["revenue_by_category"].items()))
            export_report(context["report"])
            print(f"Сохранены файлы: {SUMMARY_JSON}, {CATEGORY_CSV}, {REPORT_FILE}")

        else:
            print("Неверный пункт меню, попробуйте снова")

In [14]:
def scripted_input(answers):
    """Подменяет input заранее заданными ответами — для демонстрации консоли.

    Печатает приглашение вместе с «введённым» значением, чтобы диалог читался
    так же, как при ручном вводе.
    """
    queue = list(answers)

    def ask(prompt):
        value = queue.pop(0) if queue else "0"
        print(f"{prompt}{value}")
        return value

    return ask


# Демонстрация этапа 10: проходим по всем пунктам меню,
# включая неверный разрез ("7") и неверный пункт меню ("9")
demo_answers = ["1", "2", "7", "2", "2", "3", "4", "9", "5", "0"]
console(ask=scripted_input(demo_answers), context=context)

=== АНАЛИТИЧЕСКАЯ КОНСОЛЬ ===
1 — полный отчёт
2 — выручка по разрезу
3 — топ покупателей и товаров
4 — аномальные заказы
5 — сохранить отчёты в файлы
0 — выход
Выберите пункт меню: 1
ОТЧЕТ АНАЛИТИЧЕСКОЙ ПЛАТФОРМЫ
Обработано строк заказов: 28
Отбраковано некорректных: 5
Корректных заказов: 23
Завершенных заказов: 20

Общая выручка: 86350.00
Средний чек: 4317.50
Медианный чек: 3750.00

Выручка по категориям:
  Одежда — 37900.00
  Книги — 27000.00
  Электроника — 21450.00

Выручка по сегментам:
  опт — 63700.00
  розница — 22650.00

Топ покупателей:
  Мария Петрова — 28100.00
  Елена Морозова — 21500.00
  Ольга Смирнова — 14100.00

Аномальных заказов: 1
=== АНАЛИТИЧЕСКАЯ КОНСОЛЬ ===
1 — полный отчёт
2 — выручка по разрезу
3 — топ покупателей и товаров
4 — аномальные заказы
5 — сохранить отчёты в файлы
0 — выход
Выберите пункт меню: 2
Разрезы: 1 — категории, 2 — города, 3 — сегменты
Выберите разрез: 7
Неизвестный разрез, возвращаюсь в меню
=== АНАЛИТИЧЕСКАЯ КОНСОЛЬ ===
1 — полный отчёт
2 

In [15]:
# Для живой интерактивной работы раскомментируйте строку ниже
# и выполните ячейку: консоль будет ждать ввода с клавиатуры.

# console()

## Итоги

Конвейер отработал полностью: три источника подготовлены и загружены, пять битых строк
отбракованы с ведением журнала, источники объединены по ключу, аналитика посчитана
тремя способами с совпадающими результатами, аномалия найдена, отчёты сохранены.

Проверка созданных файлов — в ячейке ниже.

In [16]:
import os

produced = [CUSTOMERS_FILE, PRODUCTS_FILE, ORDERS_FILE, REJECTED_FILE,
            SUMMARY_JSON, CATEGORY_CSV, REPORT_FILE]

print(f"{'файл':<24}{'размер, байт':>14}")
for name in produced:
    print(f"{name:<24}{os.path.getsize(name):>14}")

print()
print("Контрольные значения:")
checks = [
    ("строк заказов", context["summary"]["total_rows"], 28),
    ("отбраковано", context["summary"]["rejected"], 5),
    ("корректных", context["summary"]["valid"], 23),
    ("завершённых", context["summary"]["completed"], 20),
    ("общая выручка", context["summary"]["total_revenue"], 86350.0),
    ("средний чек", context["summary"]["average_check"], 4317.5),
    ("медианный чек", context["summary"]["median_check"], 3750.0),
    ("аномалий", context["summary"]["anomalies"], 1),
]
for title, actual, expected in checks:
    status = "ок" if actual == expected else "РАСХОЖДЕНИЕ"
    print(f"  {title:<16} {actual:>12}   ожидалось {expected:>10}   {status}")

assert all(actual == expected for _, actual, expected in checks)
print("\nВсе контрольные значения совпали с заданием")

файл                      размер, байт
customers.csv                      621
products.csv                       522
orders.csv                         978
rejected.csv                       251
summary.json                       705
revenue_by_category.csv             87
report.txt                         770

Контрольные значения:
  строк заказов              28   ожидалось         28   ок
  отбраковано                 5   ожидалось          5   ок
  корректных                 23   ожидалось         23   ок
  завершённых                20   ожидалось         20   ок
  общая выручка         86350.0   ожидалось    86350.0   ок
  средний чек            4317.5   ожидалось     4317.5   ок
  медианный чек          3750.0   ожидалось     3750.0   ок
  аномалий                    1   ожидалось          1   ок

Все контрольные значения совпали с заданием
